# ACL Tear Detection - Combined Dataset Training

**Key improvements:**
1. **Combined dataset** with 1,202 samples (was 466)
2. **Balanced classes**: Normal=576, Partial=286, Complete=340
3. **MRI-specific normalization** (per-image)
4. **CrossEntropyLoss** with class weights
5. **More trainable layers** for medical domain adaptation

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Configuration - BINARY CLASSIFICATION (Normal vs Tear)
DATA_DIR = '/content/drive/MyDrive/dataset/combined'

# Training settings
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 2  # 0: Normal, 1: Tear (Partial + Complete)
CENTER_SLICES = 15  # Reduced to focus on true ACL region
RANDOM_SEED = 42

# Use per-image normalization (recommended for MRI)
USE_PER_IMAGE_NORM = True

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load metadata
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

print(f"Total patients: {len(metadata)}")
print(f"\nOriginal label distribution:")
print(metadata['label_name'].value_counts())

# Convert to binary: 0=Normal, 1=Tear (Partial + Complete)
metadata['label_binary'] = (metadata['label'] > 0).astype(int)
metadata['label_name_binary'] = metadata['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"\nBinary label distribution:")
print(metadata['label_name_binary'].value_counts())

# Check class balance
class_counts_df = metadata['label_binary'].value_counts().sort_index()

imbalance_ratio = class_counts_df.max() / class_counts_df.min()    print(metadata['source'].value_counts())

print(f"\nClass imbalance ratio: {imbalance_ratio:.1f}x")    print(f"\nSource distribution:")

if 'source' in metadata.columns:
# Show source distribution

In [ ]:
# Split data: patient-level split to avoid data leakage
train_df, temp_df = train_test_split(
    metadata, test_size=0.3, stratify=metadata['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print(f"Train: {len(train_df)} patients")
print(f"Val: {len(val_df)} patients")
print(f"Test: {len(test_df)} patients")

print(f"\nTrain label distribution (binary):")
print(train_df['label_name_binary'].value_counts())

In [ ]:
class ACLSliceDataset(Dataset):
    """
    Dataset that treats each slice as a separate sample.
    Uses per-image normalization for MRI data.
    """
    def __init__(self, df, data_dir, transform=None, center_slices=20):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.center_slices = center_slices

        # Pre-compute slice indices (using binary labels)
        self.samples = []
        for idx, row in self.df.iterrows():
            num_slices = int(row['num_slices'])
            start = max(0, (num_slices - center_slices) // 2)
            end = min(num_slices, start + center_slices)
            for slice_idx in range(start, end):
                self.samples.append((idx, slice_idx, int(row['label_binary'])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        patient_idx, slice_idx, label = self.samples[idx]
        row = self.df.iloc[patient_idx]

        # Load volume
        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']

        # Clamp slice_idx to valid range (in case metadata num_slices doesn't match actual volume)
        slice_idx = min(slice_idx, volume.shape[0] - 1)

        # Get slice and normalize to 0-1
        img = volume[slice_idx].astype(np.float32) / 255.0

        # Per-image normalization (KEY FIX for MRI)
        img_mean = img.mean()
        img_std = img.std() + 1e-8
        img = (img - img_mean) / img_std

        # Convert to 3-channel for pretrained models
        img = np.stack([img, img, img], axis=0)
        img = torch.from_numpy(img)

        if self.transform:
            img = self.transform(img)

        return img, label, patient_idx

    def get_labels(self):
        return [s[2] for s in self.samples]

In [ ]:
# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
])

val_transform = None

# Create datasets
train_dataset = ACLSliceDataset(train_df, DATA_DIR, train_transform, CENTER_SLICES)
val_dataset = ACLSliceDataset(val_df, DATA_DIR, val_transform, CENTER_SLICES)
test_dataset = ACLSliceDataset(test_df, DATA_DIR, val_transform, CENTER_SLICES)

print(f"Train samples (slices): {len(train_dataset)}")
print(f"Val samples (slices): {len(val_dataset)}")
print(f"Test samples (slices): {len(test_dataset)}")

In [ ]:
# Create weighted sampler for balanced training
train_labels = train_dataset.get_labels()
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

print(f"Class counts in training: {class_counts}")
print(f"Class weights: {class_weights}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Model: EfficientNet-B0 with more trainable layers
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze only first 50% of layers (more adaptation for MRI)
all_params = list(model.features.parameters())
freeze_until = len(all_params) // 2
for i, param in enumerate(all_params):
    param.requires_grad = (i >= freeze_until)

# Replace classifier
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(model.classifier[1].in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
# Loss function with class weights
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer with weight decay
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels, _) in enumerate(tqdm(loader, desc='Training')):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

In [ ]:
# Training loop with early stopping
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
patience = 10
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/dataset/best_acl_model_combined.pth')
        print(f"  -> Saved best model (val_acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_combined.png', dpi=150)
plt.show()

In [ ]:
# Load best model for evaluation
model.load_state_dict(torch.load('/content/drive/MyDrive/dataset/best_acl_model_combined.pth'))

# Final evaluation on test set
test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion, device)
print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.2f}%")

In [ ]:
# Classification Report (Slice-level)
label_names = ['Normal', 'Tear']
print("\nSlice-level Classification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

In [ ]:
# Confusion Matrix (Slice-level)
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Slice Level')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_binary.png', dpi=150)
plt.show()

# Per-class accuracy (slice-level)
print("\nPer-class accuracy (slice-level):")
for i, name in enumerate(label_names):
    class_mask = np.array(test_labels) == i
    class_acc = (np.array(test_preds)[class_mask] == i).mean() * 100
    print(f"  {name}: {class_acc:.1f}%")

print(f"\nPatient-level Accuracy: {patient_acc:.1f}%")

# ============================================patient_acc = np.mean(np.array(patient_final_preds) == np.array(patient_final_labels)) * 100

# PATIENT-LEVEL VOTING (More clinically relevant)# Patient-level accuracy

# ============================================

from collections import Counterplt.show()

plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_patient_level.png', dpi=150)

# Get patient indices from test datasetplt.title('Confusion Matrix - Patient Level (Majority Voting)')

test_patient_indices = [test_dataset.samples[i][0] for i in range(len(test_dataset))]plt.ylabel('Actual')

plt.xlabel('Predicted')

# Group predictions by patient            xticklabels=label_names, yticklabels=label_names)

patient_preds_dict = {}sns.heatmap(cm_patient, annot=True, fmt='d', cmap='Greens', 

patient_labels_dict = {}plt.figure(figsize=(8, 6))

for pred, label, patient_idx in zip(test_preds, test_labels, test_patient_indices):cm_patient = confusion_matrix(patient_final_labels, patient_final_preds)

    if patient_idx not in patient_preds_dict:# Patient-level confusion matrix

        patient_preds_dict[patient_idx] = []

        patient_labels_dict[patient_idx] = label  # All slices have same labelprint(classification_report(patient_final_labels, patient_final_preds, target_names=label_names, digits=3))

    patient_preds_dict[patient_idx].append(pred)print("\nPatient-level Classification Report:")

print(f"\nTotal test patients: {len(patient_final_preds)}")

# Majority voting per patientprint("="*50)

patient_final_preds = []print("PATIENT-LEVEL RESULTS (Majority Voting)")

patient_final_labels = []print("\n" + "="*50)

for patient_idx in sorted(patient_preds_dict.keys()):# Patient-level metrics

    preds = patient_preds_dict[patient_idx]

    # Majority vote    patient_final_labels.append(patient_labels_dict[patient_idx])

    final_pred = Counter(preds).most_common(1)[0][0]    patient_final_preds.append(final_pred)